# Step 4 (Baseline EfficientAD-S B03) — Colab GPU Training

Notebook chạy tự động toàn diện (**Runtime ▸ Run all**) để huấn luyện mô hình **EfficientAD-S (B03)** max_steps=70,000 (~8.7 epochs, ~2h trên T4 GPU) trên GPU Colab, chuẩn hóa map trên calibration good, trích xuất dự đoán thô trên calibration (460 ảnh) và fusion (306 ảnh), đóng gói artifacts và sao chép thẳng về Google Drive.

**Quy trình chuẩn bị:**
1. Nén thư mục repo thành `PCB_Lab_impl4.zip` và upload lên Google Drive (`My Drive/`).
2. Đảm bảo file dataset `DatasetVer4_Public.zip` (hoặc `datasetver4.zip`) đã có trên Google Drive (`My Drive/`).
3. Trên Colab: chọn **Runtime ▸ Change runtime type ▸ T4 GPU** (hoặc A100/L4).
4. Bấm **Runtime ▸ Run all** (hoặc chạy tuần tự từng cell).

In [ ]:
# ===== Cell 1: Gate 0 — Hardware Verification =====
import os, sys, subprocess

def banner(t):
    print('=' * 60)
    print(t)
    print('=' * 60)

banner('Gate 0: CUDA hardware check')
!nvidia-smi || echo 'nvidia-smi not found'

import torch
cuda_ok = torch.cuda.is_available()
print('torch.version:', torch.__version__)
print('torch.cuda.is_available():', cuda_ok)
if cuda_ok:
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print('GPU[0]:', name, '(compute capability', cap, ')')
    assert cap[0] >= 7, f'GPU compute capability {cap} too low for CUDA execution'
    print('CUDA compute capability OK for EfficientAD-S training')
else:
    raise SystemExit('Gate 0 FAILED: no CUDA GPU. Vui lòng chọn Runtime -> Change runtime type -> T4 GPU.')

In [ ]:
# ===== Cell 2: Dataset Setup (Auto-extract from Drive ZIP) =====
import os, shutil, zipfile, glob
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive')

possible_zips = (
    list(drive_root.glob('*[Dd]ataset*[Vv]er4*.zip')) +
    list(drive_root.glob('*[Dd]ataset*.zip')) +
    list(drive_root.glob('*[Dd]eep[Pp][Cc][Bb]*.zip'))
)

local_ds_dir = Path('/content/DatasetVer4_Public')

if possible_zips:
    ds_zip = possible_zips[0]
    print(f'--> Tìm thấy tệp ZIP dataset: {ds_zip.name}')
    local_zip = Path(f'/content/{ds_zip.name}')
    if not local_zip.exists():
        print('--> Đang sao chép dataset ZIP sang SSD local (/content)...')
        shutil.copy2(ds_zip, local_zip)
    
    temp_extract = Path('/content/ds_temp')
    if temp_extract.exists():
        shutil.rmtree(temp_extract)
    
    print('--> Đang giải nén dataset tại local...')
    with zipfile.ZipFile(local_zip, 'r') as zf:
        zf.extractall(temp_extract)
    
    manifest_files = list(temp_extract.rglob('samples.jsonl'))
    if manifest_files:
        extracted_root = manifest_files[0].parents[3]
        if local_ds_dir.exists():
            shutil.rmtree(local_ds_dir)
        shutil.move(str(extracted_root), str(local_ds_dir))
        shutil.rmtree(temp_extract, ignore_errors=True)
        if local_zip.exists():
            local_zip.unlink()
        print(f'--> Dataset sẵn sàng tại: {local_ds_dir}')
    else:
        raise FileNotFoundError('Không tìm thấy samples.jsonl trong tệp dataset ZIP giải nén!')
else:
    if not local_ds_dir.exists():
        raise FileNotFoundError('Không tìm thấy tệp ZIP dataset trên Drive!')

os.environ['DATASET_ROOT'] = str(local_ds_dir)

In [ ]:
# ===== Cell 3: Codebase Setup (Auto-extract from Drive ZIP) =====
import os, shutil, zipfile
from pathlib import Path

drive_root = Path('/content/drive/MyDrive')
possible_repo_zips = (
    list(drive_root.glob('*[Pp][Cc][Bb]*[Ii]mpl4*.zip')) +
    list(drive_root.glob('*[Pp][Cc][Bb]*[Ll]ab*.zip'))
)

local_repo_dir = Path('/content/PCB_Lab_impl4')

if possible_repo_zips:
    repo_zip = possible_repo_zips[0]
    print(f'--> Tìm thấy tệp ZIP repo: {repo_zip.name}')
    local_zip = Path(f'/content/{repo_zip.name}')
    if not local_zip.exists():
        print('--> Đang sao chép repo ZIP sang SSD local (/content)...')
        shutil.copy2(repo_zip, local_zip)
    
    temp_extract = Path('/content/repo_temp')
    if temp_extract.exists():
        shutil.rmtree(temp_extract)
    
    print('--> Đang giải nén repo tại local...')
    with zipfile.ZipFile(local_zip, 'r') as zf:
        zf.extractall(temp_extract)
    
    plan_files = list(temp_extract.rglob('PLAN.md'))
    if plan_files:
        extracted_root = plan_files[0].parent
        if local_repo_dir.exists():
            shutil.rmtree(local_repo_dir)
        shutil.move(str(extracted_root), str(local_repo_dir))
        shutil.rmtree(temp_extract, ignore_errors=True)
        if local_zip.exists():
            local_zip.unlink()
        print(f'--> Repo sẵn sàng tại: {local_repo_dir}')
    else:
        raise FileNotFoundError('Không tìm thấy PLAN.md trong tệp repo ZIP giải nén!')
else:
    if not local_repo_dir.exists():
        raise FileNotFoundError('Không tìm thấy tệp ZIP repo trên Drive!')

os.chdir(str(local_repo_dir))
print('Thư mục hiện tại:', os.getcwd())

In [ ]:
# ===== Cell 4: Dependencies Installation =====
import subprocess, sys

print('--> Cài đặt PyTorch CUDA wheels và requirements cho Step 4...')
!pip install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124
!pip install -r requirements/step4.txt

import torch, cv2, numpy
print('PyTorch version:', torch.__version__)
print('CUDA version:', torch.version.cuda)
print('Device name:', torch.cuda.get_device_name(0))

In [ ]:
# ===== Cell 5: Train EfficientAD-S (B03) — max_steps=70,000 (~8.7 Epochs) =====
!python scripts/train_efficientad.py --config configs/models/efficientad_s.yaml --seed 42 --device 0 --allow-download --max-steps 70000

In [ ]:
# ===== Cell 6: Extract Predictions (Calibration & Fusion) =====
!python scripts/extract_efficientad_predictions.py --artifact artifacts/efficientad/B03_efficientad_s/seed42 --device 0

In [ ]:
# ===== Cell 7: Verify Prediction Lines & Artifacts =====
import json
from pathlib import Path

artifact = Path('artifacts/efficientad/B03_efficientad_s/seed42')
assert artifact.exists(), 'Thư mục artifact không tồn tại!'

meta = json.loads((artifact / 'artifact.json').read_text(encoding='utf-8'))
print('Model ID:', meta['model_id'])
print('Checkpoint SHA256:', meta['checkpoint_sha256'])
print('Smoke flag:', meta['smoke'])
assert meta['smoke'] is False, 'Artifact không được phép là smoke artifact!'

for split, expected in [('calibration', 460), ('fusion', 306)]:
    pred_file = artifact / f'preds_{split}.jsonl'
    assert pred_file.exists(), f'Thiếu file {pred_file}!'
    lines = pred_file.read_text(encoding='utf-8').splitlines()
    rows = [json.loads(line) for line in lines]
    assert len(rows) == expected, f'Số dòng {split} ({len(rows)}) không khớp kỳ vọng {expected}!'
    assert len({r['sample_id'] for r in rows}) == expected, f'Trùng lặp sample_id trong {split}!'
    assert all(r['error_reason'] is None for r in rows), f'Có ảnh bị lỗi inference trong {split}!'
    print(f'--> {split.capitalize()}: {len(rows)}/{expected} rows OK, 0 error.')

print('\n>>> TOÀN BỘ ARTIFACT VÀ DỰ ĐOÁN ĐẠT CHUẨN 100% <<<')

In [ ]:
# ===== Cell 8: Package Artifacts & Export to Drive =====
import shutil
from pathlib import Path
from google.colab import files

print('--> Đang đóng gói artifacts và runs...')
!zip -r /content/efficientad_step4_artifacts.zip artifacts/efficientad/
!zip -r /content/efficientad_step4_runs.zip runs/efficientad/

drive_target = Path('/content/drive/MyDrive')
if drive_target.exists():
    print('--> Đang sao chép các tệp ZIP về Google Drive...')
    shutil.copy2('/content/efficientad_step4_artifacts.zip', drive_target / 'efficientad_step4_artifacts.zip')
    shutil.copy2('/content/efficientad_step4_runs.zip', drive_target / 'efficientad_step4_runs.zip')
    print('--> Đã lưu thành công vào Drive:')
    print('   - /content/drive/MyDrive/efficientad_step4_artifacts.zip')
    print('   - /content/drive/MyDrive/efficientad_step4_runs.zip')

print('\nĐang kích hoạt tải về máy local qua trình duyệt...')
try:
    files.download('/content/efficientad_step4_artifacts.zip')
    files.download('/content/efficientad_step4_runs.zip')
except Exception as e:
    print('Không thể kích hoạt tự động download, bạn có thể lấy trực tiếp từ Drive:', e)